# Three Body Gravitational Simulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Body as objects of a Class

## Initial conditions

In [153]:
#This block read the inital condition file and stores the data in a single array
#first three items within the array are mass, position vector and velocity vector of the first body. 
# init_cond[0][0] would refer to the mass of the first body
# init_cond[0][1] would refer to the position vector (xyz) of the first body
# init_cond[0][2] would refer to the position vector (xyz) of the first body
# init_cond[0][2][1] would refer to the y component of the velocity vector of the first body
# init_cond[1][1] would refer to the position vector (xyz) of the first body

#[M] = Kg    [V]=m/s     [t] = s.   [x,y,z] = m

init_cond=[]
n_bodies = 0
with open("init.txt") as f:
    next(f)
    for line in f:
        values = line.split()
        #print(values)
        mass = float(values[1])
        pos = [float(i) for i in values[2:5]]
        vel = [float(i) for i in values[5:]]
        init_cond.append((mass,pos,vel))
        n_bodies += 1

print(init_cond)
print(init_cond[0][2][1]*2)
print(n_bodies)

[(0.5, [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]), (1.5, [0.0, 1.0, 0.0], [0.0, 0.0, 0.0]), (2.5, [1.0, 0.0, 0.0], [0.0, 0.0, 0.0])]
0.0
3


In [154]:
def force_calc(init_cond):

    forces_xyz = []
    forces_mag = []
    G = 6.6743e-11
    for body in range(n_bodies):

        total_Fx = 0.
        total_Fy = 0.
        total_Fz = 0.
        total_Fmag = 0.

        for other in range(n_bodies):

            if body == other:
                pass
            #print(body,"-", other)

            dx = init_cond[other][1][0] - init_cond[body][1][0]
            dy = init_cond[other][1][1] - init_cond[body][1][1]
            dz = init_cond[other][1][2] - init_cond[body][1][2]

            #print(init_cond[other][1][0], "-", init_cond[body][1][0])
            #print(dx,dy,dz)

            distance = np.sqrt(dx**2 + dy**2 + dz**2)

            try:
                Fx = G * init_cond[other][0] * init_cond[body][0] / dx**2
            except:
                Fx = 0.0
            try:
                Fy = G * init_cond[other][0] * init_cond[body][0] / dy**2
            except:
                Fy = 0.0
            try:
                Fz = G * init_cond[other][0] * init_cond[body][0] / dz**2
            except:
                Fz = 0.0

            F_mag = G * init_cond[other][0] * init_cond[body][0] / distance**2

            total_Fx += Fx
            total_Fy += Fy
            total_Fz += Fz
            total_Fmag += F_mag


            #Forces are stored in logical orther, 0-1, 0-2, 0-3, 0-4, 1-2, 1-3, 1-4, 2-3, 2-4 ...

        forces_xyz.append((total_Fx,total_Fy,total_Fz))
        forces_mag.append(total_Fmag)
    
    return forces_xyz, forces_mag
        

forces_xyz, forces_mag = force_calc(init_cond)
forces_xyz


/var/folders/z0/9cg8kpjn3c5dpyv43k19nh_40000gn/T/ipykernel_8389/3430768991.py:41: RuntimeWarning: divide by zero encountered in scalar divide
  F_mag = G * init_cond[other][0] * init_cond[body][0] / distance**2


[(8.342874999999999e-11, 5.0057249999999995e-11, 0.0),
 (2.5028624999999996e-10, 3.0034349999999995e-10, 0.0),
 (3.3371499999999997e-10, 2.5028624999999996e-10, 0.0)]

In [155]:
time_step = 1
next_cond = init_cond
print(next_cond)

def velocity_update(init_cond,forces_xyz,time_step):

    vel_xyz = []


    for body in range(n_bodies):

        vel_x = init_cond[body][2][0] + forces_xyz[body][0] / init_cond[body][0] * time_step
        vel_y = init_cond[body][2][1] + forces_xyz[body][1] / init_cond[body][0] * time_step
        vel_z = init_cond[body][2][2] + forces_xyz[body][2] / init_cond[body][0] * time_step

        next_cond[body][2][0] = vel_x
        next_cond[body][2][1] = vel_y
        next_cond[body][2][2] = vel_z

        vel_xyz.append((vel_x,vel_y,vel_z))

    return next_cond, vel_xyz

next_cond, vel_xyz = velocity_update(init_cond, forces_xyz, time_step)

print(next_cond)

print("\n",vel_xyz)

[(0.5, [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]), (1.5, [0.0, 1.0, 0.0], [0.0, 0.0, 0.0]), (2.5, [1.0, 0.0, 0.0], [0.0, 0.0, 0.0])]
[(0.5, [0.0, 0.0, 0.0], [1.6685749999999998e-10, 1.0011449999999999e-10, 0.0]), (1.5, [0.0, 1.0, 0.0], [1.6685749999999998e-10, 2.0022899999999996e-10, 0.0]), (2.5, [1.0, 0.0, 0.0], [1.33486e-10, 1.0011449999999999e-10, 0.0])]

 [(1.6685749999999998e-10, 1.0011449999999999e-10, 0.0), (1.6685749999999998e-10, 2.0022899999999996e-10, 0.0), (1.33486e-10, 1.0011449999999999e-10, 0.0)]


In [156]:
def position_update(next_cond,time_step):
    print(next_cond)

    for body in range(n_bodies):

        next_cond[body][1][0] += vel_xyz[body][0] * time_step
        next_cond[body][1][1] += vel_xyz[body][1] * time_step
        next_cond[body][1][2] += vel_xyz[body][2] * time_step

    return next_cond

next_cond = position_update(next_cond,time_step)
print(next_cond)


[(0.5, [0.0, 0.0, 0.0], [1.6685749999999998e-10, 1.0011449999999999e-10, 0.0]), (1.5, [0.0, 1.0, 0.0], [1.6685749999999998e-10, 2.0022899999999996e-10, 0.0]), (2.5, [1.0, 0.0, 0.0], [1.33486e-10, 1.0011449999999999e-10, 0.0])]
[(0.5, [1.6685749999999998e-10, 1.0011449999999999e-10, 0.0], [1.6685749999999998e-10, 1.0011449999999999e-10, 0.0]), (1.5, [1.6685749999999998e-10, 1.000000000200229, 0.0], [1.6685749999999998e-10, 2.0022899999999996e-10, 0.0]), (2.5, [1.000000000133486, 1.0011449999999999e-10, 0.0], [1.33486e-10, 1.0011449999999999e-10, 0.0])]
